<a href="https://colab.research.google.com/github/HTKhuongNinh-FPTU/Machine-Learning-Doc/blob/main/SE196288_HaThucKhuongNinh_Lab01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
mport json
import re
from datetime import datetime
from collections import Counter
from google.colab import files

uploaded = files.upload()

with open("THONG_TIN_KHACH_HANG.json", encoding="utf-8") as f:
    data = json.load(f)

print(f"Số bản ghi: {len(data)}")
print("Ví dụ bản ghi đầu tiên:")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

Saving THONG_TIN_KHACH_HANG.json to THONG_TIN_KHACH_HANG (3).json
Số bản ghi: 1000
Ví dụ bản ghi đầu tiên:
{
  "Họ và tên": "Trần Long Tân",
  "Số điện thoại": "0392 768 040",
  "Email": "trantan1990@hotmail.com",
  "Địa chỉ": "Số 367, Đường Đinh Lễ, Huyện Củ Chi, Hà Tĩnh",
  "Ngày sinh": "09/10/1998",
  "Mã KH": "CUS250001",
  "Thời gian đăng nhập": "2024-06-23 08:09",
  "Người phụ trách": "Ngô Văn Phúc",
  "Ghi chú": "Cần gọi lại"
}


**Phần 1**

In [ ]:
def is_valid_phone(phone_str: str) -> bool:
    digits_only = re.sub(r'\D', '', phone_str)

    if digits_only.startswith('84'):
        digits_only = '0' + digits_only[2:]

    pattern = r'^0[35789][0-9]{8}$'
    return bool(re.match(pattern, digits_only))
def normalize_phone(phone_str: str) -> str:
    digits_only = re.sub(r'\D', '', phone_str)

    if digits_only.startswith('84'):
        digits_only = '0' + digits_only[2:]

    if is_valid_phone(phone_str):
        return digits_only
    else:
        return None
print("=" * 70)
print("KIỂM THỬ")
print("=" * 70)

test_cases = [
    "0392 768 040",
    "+84 91 234 5678",
    "84912345678",
    "(+84)912345678",
    "085-713-9656",
    "0973.204.190",
    "12345",
    "abcdefghij",]

for t in test_cases:
    print(f"{t:25s} → valid={is_valid_phone(t)}, normalized={normalize_phone(t)}")
with open("THONG_TIN_KHACH_HANG.json", encoding="utf-8") as f:
    data = json.load(f)
valid_count = 0
invalid_count = 0
normalized_phones = []
invalid_records = []

for record in data:
    phone = record.get("Số điện thoại", "")
    normalized = normalize_phone(phone)

    if normalized:
        valid_count += 1
        normalized_phones.append(normalized)
    else:
        invalid_count += 1
        invalid_records.append({
            "Mã KH": record.get("Mã KH"),
            "Họ và tên": record.get("Họ và tên"),
            "SĐT gốc": phone})

print(f"\nSố lượng SĐT hợp lệ:       {valid_count}")
print(f"Số lượng SĐT không hợp lệ: {invalid_count}")
if invalid_records:
    print(f"\n" + "-" * 70)
    print("DANH SÁCH SĐT KHÔNG HỢP LỆ")
    print("-" * 70)
    print(f"\n{'Mã KH':<12} {'Họ và tên':<25} {'SĐT gốc':<20}")
    print("-" * 70)

    for record in invalid_records[:10]:
        print(f"{record['Mã KH']:<12} {record['Họ và tên']:<25} {record['SĐT gốc']:<20}")

    if len(invalid_records) > 10:
        print(f"... và {len(invalid_records) - 10} cái khác")

print(f"\n" + "-" * 70)
print("THỐNG KÊ ĐẦU SỐ")
print("-" * 70)

if normalized_phones:
    prefixes = [phone[:3] for phone in normalized_phones]
    prefix_counts = Counter(prefixes)

    print(f"\n{'Đầu số':<10} {'Số lượng':<12}")
    print("-" * (10 + 1 + 12))

    for prefix, count in prefix_counts.most_common():
        print(f"{prefix:<10} {count:<12}")

    top_prefix, top_count = prefix_counts.most_common(1)[0]

print(f"Đầu số phổ biến nhất: {top_prefix} ({top_count} thuê bao)")

KIỂM THỬ
0392 768 040              → valid=True, normalized=0392768040
+84 91 234 5678           → valid=True, normalized=0912345678
84912345678               → valid=True, normalized=0912345678
(+84)912345678            → valid=True, normalized=0912345678
085-713-9656              → valid=True, normalized=0857139656
0973.204.190              → valid=True, normalized=0973204190
12345                     → valid=False, normalized=None
abcdefghij                → valid=False, normalized=None

Số lượng SĐT hợp lệ:       1000
Số lượng SĐT không hợp lệ: 0

----------------------------------------------------------------------
THỐNG KÊ ĐẦU SỐ
----------------------------------------------------------------------

Đầu số     Số lượng    
-----------------------
081        45          
079        41          
076        40          
097        37          
038        37          
090        37          
078        37          
037        37          
083        36          
039        35      

In [ ]:
def is_valid_email(email: str) -> bool:
    if not isinstance(email, str):
        return False
    email = email.strip()
    pattern = r'^[a-zA-Z0-9_,-]+(?:\.[a-zA-Z0-9_,-]+)*@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,6}$'
    return bool(re.match(pattern, email))
def extract_domain(email: str) -> str:
    if not is_valid_email(email):
        return "Invalid Email"
    try:
        return email.strip().split('@')[1]
    except IndexError:
        return ""
def analyze_email_lab(data):
    invalid_emails = []
    domains = []
    for record in data:
        email = record.get("Email", "")
        if is_valid_email(email):
            domains.append(extract_domain(email))
        else:
            invalid_emails.append(email)
    domain_counts = Counter(domains)
    most_common_domain = domain_counts.most_common(1)[0] if domains else ("None", 0)

    print("\n=== THỐNG KÊ EMAIL ===")
    print(f"- Danh sách email không hợp lệ: {invalid_emails if invalid_emails else 'Không có'}")
    print(f"- Domain xuất hiện nhiều nhất: {most_common_domain[0]} ({most_common_domain[1]} lần)")

analyze_email_lab(data)


=== THỐNG KÊ EMAIL ===
- Danh sách email không hợp lệ: Không có
- Domain xuất hiện nhiều nhất: outlook.com (151 lần)


In [ ]:
def parse_address(addr: str) -> dict:
    result = {
        "so_nha": "",
        "duong": "",
        "quan_huyen": "",
        "tinh_thanh": "",
        "tang": "",
        "can": ""}
    if not isinstance(addr, str) or not addr:
        return result
    parts = [p.strip() for p in addr.split(',')]
    if len(parts) >= 1:
        result["tinh_thanh"] = parts[-1]
    if len(parts) >= 2:
        result["quan_huyen"] = parts[-2]
    remainder_parts = parts[:-2] if len(parts) >= 2 else parts[:-1]
    remainder_str = ", ".join(remainder_parts)
    floor_match = re.search(r'Tầng\s+(\d+|\w+)', remainder_str, re.IGNORECASE)
    room_match = re.search(r'Căn\s+(\d+|\w+)', remainder_str, re.IGNORECASE)
    if floor_match:
        result["tang"] = floor_match.group(1)
    if room_match:
        result["can"] = room_match.group(1)
    clean_street_str = re.sub(r'(Tầng\s+\d+|Căn\s+\d+)[,\s]*', '', remainder_str, flags=re.IGNORECASE).strip()
    house_num_pattern = r'^(?:Số\s+)?(?P<so_nha>\d+(?:/\d+)?(?:\s*[A-Za-z])?)\s+(?:Đường\s+)?(?P<duong>.*)$'
    match_house = re.match(house_num_pattern, clean_street_str, re.IGNORECASE)
    if match_house:
        result["so_nha"] = match_house.group("so_nha")
        result["duong"] = match_house.group("duong")
    else:
        result["duong"] = clean_street_str
    return result

test_cases = [
    "367 Đinh Lễ, Huyện Củ Chi, Hà Tĩnh",
    "Số 24, Đường Tràng Thi, Quận Phú Nhuận, Quảng Trị",
    "138/7 Trần Hưng Đạo, Quận Tân Bình, Trà Vinh",
    "Tầng 5, Căn 03, 100 Lê Lợi, Quận 1, TP. Hồ Chí Minh"]

import json
print("--- KẾT QUẢ PARSE ĐỊA CHỈ ---")
for tc in test_cases:
    print(f"\nĐịa chỉ gốc: '{tc}'")
    print(json.dumps(parse_address(tc), ensure_ascii=False, indent=2))

--- KẾT QUẢ PARSE ĐỊA CHỈ ---

Địa chỉ gốc: '367 Đinh Lễ, Huyện Củ Chi, Hà Tĩnh'
{
  "so_nha": "367",
  "duong": "Đinh Lễ",
  "quan_huyen": "Huyện Củ Chi",
  "tinh_thanh": "Hà Tĩnh",
  "tang": "",
  "can": ""
}

Địa chỉ gốc: 'Số 24, Đường Tràng Thi, Quận Phú Nhuận, Quảng Trị'
{
  "so_nha": "",
  "duong": "Số 24, Đường Tràng Thi",
  "quan_huyen": "Quận Phú Nhuận",
  "tinh_thanh": "Quảng Trị",
  "tang": "",
  "can": ""
}

Địa chỉ gốc: '138/7 Trần Hưng Đạo, Quận Tân Bình, Trà Vinh'
{
  "so_nha": "138/7",
  "duong": "Trần Hưng Đạo",
  "quan_huyen": "Quận Tân Bình",
  "tinh_thanh": "Trà Vinh",
  "tang": "",
  "can": ""
}

Địa chỉ gốc: 'Tầng 5, Căn 03, 100 Lê Lợi, Quận 1, TP. Hồ Chí Minh'
{
  "so_nha": "100",
  "duong": "Lê Lợi",
  "quan_huyen": "Quận 1",
  "tinh_thanh": "TP. Hồ Chí Minh",
  "tang": "5",
  "can": "03"
}


In [ ]:
def parse_address(addr: str) -> dict:
    result = {
        "so_nha": "",
        "duong": "",
        "quan_huyen": "",
        "tinh_thanh": "",
        "tang": "",
        "can": ""}
    if not isinstance(addr, str) or not addr:
        return result
    parts = [p.strip() for p in addr.split(',')]
    if len(parts) >= 1:
        result["tinh_thanh"] = parts[-1]
    if len(parts) >= 2:
        result["quan_huyen"] = parts[-2]
    remainder_parts = parts[:-2] if len(parts) >= 2 else parts[:-1]
    remainder_str = ", ".join(remainder_parts)

    floor_match = re.search(r'Tầng\s+(\d+|\w+)', remainder_str, re.IGNORECASE)
    room_match = re.search(r'Căn\s+(\d+|\w+)', remainder_str, re.IGNORECASE)
    if floor_match:
        result["tang"] = floor_match.group(1)
    if room_match:
        result["can"] = room_match.group(1)

    clean_street_str = re.sub(r'(Tầng\s+\d+|Căn\s+\d+)[,\s]*', '', remainder_str, flags=re.IGNORECASE).strip()
    house_num_pattern = r'^(?:Số\s+)?(?P<so_nha>\d+(?:/\d+)?(?:\s*[A-Za-z])?)\s+(?:Đường\s+)?(?P<duong>.*)$'
    match_house = re.match(house_num_pattern, clean_street_str, re.IGNORECASE)
    if match_house:
        result["so_nha"] = match_house.group("so_nha")
        result["duong"] = match_house.group("duong")
    else:
        result["duong"] = clean_street_str
    return result
if __name__ == "__main__":
    print("--- 1. KIỂM THỬ HÀM PARSE_ADDRESS ---")
    test_cases = [
        "367 Đinh Lễ, Huyện Củ Chi, Hà Tĩnh",
        "138/7 Trần Hưng Đạo, Quận Tân Bình, Trà Vinh",
        "Tầng 5, Căn 03, 100 Lê Lợi, Quận 1, TP. Hồ Chí Minh"]
    for tc in test_cases:
        print(f"\nĐịa chỉ gốc: '{tc}'")
        print(json.dumps(parse_address(tc), ensure_ascii=False, indent=2))
    print("\n" + "="*60 + "\n")
    tinh_thanh_list = []
    for record in data:
        address_str = record.get("Địa chỉ", "")
        parsed = parse_address(address_str)
        if parsed["tinh_thanh"]:
            tinh_thanh_list.append(parsed["tinh_thanh"])
    thong_ke_tinh = Counter(tinh_thanh_list)
    most_common = thong_ke_tinh.most_common(1)
    if most_common:
        print(f"Tỉnh/Thành phố có nhiều khách hàng nhất: {most_common[0][0]} ({most_common[0][1]} khách hàng).")


--- 1. KIỂM THỬ HÀM PARSE_ADDRESS VỚI CÁC ĐỊNH DẠNG MẪU ---

Địa chỉ gốc: '367 Đinh Lễ, Huyện Củ Chi, Hà Tĩnh'
{
  "so_nha": "367",
  "duong": "Đinh Lễ",
  "quan_huyen": "Huyện Củ Chi",
  "tinh_thanh": "Hà Tĩnh",
  "tang": "",
  "can": ""
}

Địa chỉ gốc: '138/7 Trần Hưng Đạo, Quận Tân Bình, Trà Vinh'
{
  "so_nha": "138/7",
  "duong": "Trần Hưng Đạo",
  "quan_huyen": "Quận Tân Bình",
  "tinh_thanh": "Trà Vinh",
  "tang": "",
  "can": ""
}

Địa chỉ gốc: 'Tầng 5, Căn 03, 100 Lê Lợi, Quận 1, TP. Hồ Chí Minh'
{
  "so_nha": "100",
  "duong": "Lê Lợi",
  "quan_huyen": "Quận 1",
  "tinh_thanh": "TP. Hồ Chí Minh",
  "tang": "5",
  "can": "03"
}


Tỉnh/Thành phố có nhiều khách hàng nhất: Hà Nam (32 khách hàng).


**Phần 2**

In [ ]:
def normalize_date(date_str: str) -> str:
    if not date_str or not isinstance(date_str, str):
        return None
    date_str = date_str.strip()
    if re.match(r"^\d{4}-\d{2}-\d{2}$", date_str):
        return date_str
    match_dot = re.match(r"^(\d{1,2})\.(\d{1,2})\.(\d{4})$", date_str)
    if match_dot:
        d, m, y = match_dot.groups()
        return f"{int(y):04d}-{int(m):02d}-{int(d):02d}"
    match_dash = re.match(r"^(\d{1,2})-(\d{1,2})-(\d{4})$", date_str)
    if match_dash:
        d, m, y = match_dash.groups()
        return f"{int(y):04d}-{int(m):02d}-{int(d):02d}"
    match_slash = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", date_str)
    if match_slash:
        p1, p2, y = match_slash.groups()
        val1, val2 = int(p1), int(p2)
        if val1 > 12:
            return f"{int(y):04d}-{val2:02d}-{val1:02d}"
        elif val2 > 12:
            return f"{int(y):04d}-{val1:02d}-{val2:02d}"
        else:
            return f"{int(y):04d}-{val1:02d}-{val2:02d}"
    return None
def get_age(date_str: str) -> int:
    normalized = normalize_date(date_str)
    if not normalized:
        return -1
    try:
        birth_year = int(normalized.split("-")[0])
        current_year = 2026
        return current_year - birth_year
    except Exception:
        return -1
if __name__ == "__main__":
    file_path = "THONG_TIN_KHACH_HANG.json"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        age_groups = {
            "Dưới 18 tuổi": 0,
            "18–22 tuổi": 0,
            "23–30 tuổi": 0,
            "Trên 30 tuổi": 0,
            "Không xác định": 0,}
        for record in data:
            raw_date = record.get("Ngày sinh", "")
            age = get_age(raw_date)

            if age == -1:
                age_groups["Không xác định"] += 1
            elif age < 18:
                age_groups["Dưới 18 tuổi"] += 1
            elif 18 <= age <= 22:
                age_groups["18–22 tuổi"] += 1
            elif 23 <= age <= 30:
                age_groups["23–30 tuổi"] += 1
            else:
                age_groups["Trên 30 tuổi"] += 1

        print("=== BÁO CÁO THỐNG KÊ PHÂN BỐ ĐỘ TUỔI KHÁCH HÀNG ===")
        for group, count in age_groups.items():
            print(f" * {group:15s}: {count} khách hàng")

=== BÁO CÁO THỐNG KÊ PHÂN BỐ ĐỘ TUỔI KHÁCH HÀNG ===
 * Dưới 18 tuổi   : 0 khách hàng
 * 18–22 tuổi     : 62 khách hàng
 * 23–30 tuổi     : 230 khách hàng
 * Trên 30 tuổi   : 708 khách hàng
 * Không xác định : 0 khách hàng


In [ ]:
import unicodedata
def normalize_name(name: str) -> str:
    if not name or not isinstance(name, str):
        return ""
    lowercase_vietnamese_chars = 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễđìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹ'
    uppercase_vietnamese_chars = ''.join(c.upper() for c in lowercase_vietnamese_chars)

    allowed_chars_pattern = f"a-zA-Z{lowercase_vietnamese_chars}{uppercase_vietnamese_chars}\\s"

    cleaned = re.sub(
        f"[^({allowed_chars_pattern})]",
        "",
        name,)
    words = cleaned.split()
    words = [w.capitalize() for w in words]
    return " ".join(words)
def split_name(name: str) -> dict:
    result = {"ho": "", "ten_dem": "", "ten": ""}
    name_clean = normalize_name(name)
    if not name_clean:
        return result
    parts = name_clean.split()
    num_parts = len(parts)
    if num_parts == 1:
        result["ten"] = parts[0]
    elif num_parts == 2:
        result["ho"] = parts[0]
        result["ten"] = parts[1]
    elif num_parts > 2:
        result["ho"] = parts[0]
        result["ten"] = parts[-1]
        result["ten_dem"] = " ".join(parts[1:-1])
    return result
def remove_diacritics(name: str) -> str:
    name_clean = normalize_name(name)
    if not name_clean:
        return ""
    name_clean = name_clean.replace("Đ", "D").replace("đ", "d")
    nfkd_form = unicodedata.normalize("NFKD", name_clean)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])
if __name__ == "__main__":
    file_path = "THONG_TIN_KHACH_HANG.json"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        count_2_parts = 0
        example_list = []
        for record in data:
            fullname = record.get("Họ và tên", "")
            words = normalize_name(fullname).split()
            if len(words) == 2:
                count_2_parts += 1
                if len(example_list) < 3:
                    example_list.append(fullname)

        print("=== BÁO CÁO THỐNG KÊ CHUẨN HÓA HỌ TÊN ===")
        print(f"Tổng số khách hàng trong dataset: {len(data)}")
        print(f"Số lượng người CHỈ CÓ 2 PHẦN (Họ + Tên, không có tên đệm): {count_2_parts} người")

        if example_list:
            print("\nMột số ví dụ thực tế tìm thấy:")
            for idx, name in enumerate(example_list, 1):
                print(f"  {idx}. Gốc: '{name}' \n     -> Chuẩn hóa: '{normalize_name(name)}' \n     -> Không dấu: '{remove_diacritics(name)}'")


=== BÁO CÁO THỐNG KÊ CHUẨN HÓA HỌ TÊN ===
Tổng số khách hàng trong dataset: 1000
Số lượng người CHỈ CÓ 2 PHẦN (Họ + Tên, không có tên đệm): 39 người

Một số ví dụ thực tế tìm thấy:
  1. Gốc: 'Khưu Thanh' 
     -> Chuẩn hóa: 'Khưu Thanh' 
     -> Không dấu: 'Khuu Thanh'
  2. Gốc: 'Hoàng Cẩm' 
     -> Chuẩn hóa: 'Hoàng Cẩm' 
     -> Không dấu: 'Hoang Cam'
  3. Gốc: 'Đào Uyên' 
     -> Chuẩn hóa: 'Đào Uyên' 
     -> Không dấu: 'Dao Uyen'


In [ ]:
def normalize_ma_kh(ma: str) -> str:
    if not ma or not isinstance(ma, str):
        return None
    match = re.search(r"(\d+)$", ma.strip())
    if match:
        number_str = match.group(1)

        if len(number_str) > 3 and number_str[:2] in ["22","23","24","25",]:
            stt = int(number_str[2:])
        else:
            stt = int(number_str)
        return f"KH-{stt:05d}"
    return None
def normalize_datetime(dt_str: str) -> str:
    if not dt_str or not isinstance(dt_str, str):
        return None
    dt_str = dt_str.strip()
    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%Y/%m/%d %H:%M:%S",
        "%Y/%m/%d %H:%M",
        "%d-%m-%Y %H:%M:%S",
        "%d-%m-%Y %H:%M",
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y %H:%M",
        "%d/%m/%Y %I:%M:%S %p",
        "%d/%m/%Y %I:%M %p",]
    for fmt in formats:
        try:
            dt_obj = datetime.strptime(dt_str, fmt)
            return dt_obj.strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            continue
    return None
if __name__ == "__main__":
    file_path = "THONG_TIN_KHACH_HANG.json"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        staff_counts = {}
        for record in data:
            staff_name = record.get("Người phụ trách", "")
            if staff_name:
                staff_name = staff_name.strip()
                staff_counts[staff_name] = staff_counts.get(staff_name, 0) + 1

        max_customers = max(staff_counts.values())
        min_customers = min(staff_counts.values())

        most_busy_staff = [name for name, count in staff_counts.items() if count == max_customers]
        least_busy_staff = [name for name, count in staff_counts.items() if count == min_customers]

        print("=== BÁO CÁO THỐNG KÊ NHÂN VIÊN PHỤ TRÁCH ===")
        print(f"Tổng số nhân viên trong danh sách: {len(staff_counts)} người")

        print(f"\n* Nhân viên phụ trách NHIỀU khách hàng nhất ({max_customers} KH):")
        for staff in most_busy_staff:
            print(f"  - {staff}")

        print(f"\n* Nhân viên phụ trách ÍT khách hàng nhất ({min_customers} KH):")
        for staff in least_busy_staff:
            print(f"  - {staff}")

        print("-" * 60)

        print("=== KIỂM THỬ CHUẨN HÓA MÃ KH ===")
        ma_kh_test_cases = {
            "CUS250001": "KH-00001",
            "MKH-123": "KH-00123",
            "INVALID_ID": None}
        for raw_ma, expected in ma_kh_test_cases.items():
            normalized_ma = normalize_ma_kh(raw_ma)
            print(f"  Mã gốc: '{raw_ma}' -> Chuẩn hóa: '{normalized_ma}'")

        print("\n" + "-" * 60)
        print("=== KIỂM THỬ CHUẨN HÓA THỜI GIAN ĐĂNG NHẬP ===")
        datetime_test_cases = {
            "2024-06-23 08:09:00": "2024-06-23 08:09:00",
            "18/07/2024 10:42:30 AM": "2024-07-18 10:42:30",
            "Invalid Date": None}
        for raw_dt, expected in datetime_test_cases.items():
            normalized_dt = normalize_datetime(raw_dt)
            print(f"  Thời gian gốc: '{raw_dt}' -> Chuẩn hóa: '{normalized_dt}'")

=== BÁO CÁO THỐNG KÊ NHÂN VIÊN PHỤ TRÁCH ===
Tổng số nhân viên trong danh sách: 20 người

* Nhân viên phụ trách NHIỀU khách hàng nhất (60 KH):
  - Hoàng Đức Mạnh

* Nhân viên phụ trách ÍT khách hàng nhất (41 KH):
  - Đinh Thị Thảo
  - Lý Tuấn Anh
  - Trần Văn Bình
------------------------------------------------------------
=== KIỂM THỬ CHUẨN HÓA MÃ KH ===
  Mã gốc: 'CUS250001' -> Chuẩn hóa: 'KH-00001'
  Mã gốc: 'MKH-123' -> Chuẩn hóa: 'KH-00123'
  Mã gốc: 'INVALID_ID' -> Chuẩn hóa: 'None'

------------------------------------------------------------
=== KIỂM THỬ CHUẨN HÓA THỜI GIAN ĐĂNG NHẬP ===
  Thời gian gốc: '2024-06-23 08:09:00' -> Chuẩn hóa: '2024-06-23 08:09:00'
  Thời gian gốc: '18/07/2024 10:42:30 AM' -> Chuẩn hóa: '2024-07-18 10:42:30'
  Thời gian gốc: 'Invalid Date' -> Chuẩn hóa: 'None'


**Phần 3**

In [ ]:
def edit_distance(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j],dp[i][j - 1],dp[i - 1][j - 1])
    return dp[m][n]
def similarity_score(s1: str, s2: str) -> float:
    if not s1 and not s2:
        return 1.0
    ed = edit_distance(s1, s2)
    max_len = max(len(s1), len(s2))

    return 1.0 - (ed / max_len)
if __name__ == "__main__":
    print("--- ĐANG CHẠY CÁC TEST CASES KIỂM TRA ---")
    assert edit_distance("kitten", "sitting") == 3
    assert edit_distance("sunday", "saturday") == 3
    assert edit_distance("", "abc") == 3
    assert edit_distance("abc", "") == 3
    assert edit_distance("abc", "abc") == 0
    assert edit_distance("Nguyen", "Nguyên") == 1
    print("Tất cả test cases phần (a) đều PASS!")
    print("-" * 50)
    print("Minh họa kết quả tính độ tương đồng similarity_score(s1, s2):")
    test_pairs = [
        ("kitten", "sitting"),
        ("sunday", "saturday"),
        ("Nguyen", "Nguyên"),
        ("abc", "abc"),
        ("", "abc")]

    for str1, str2 in test_pairs:
        ed = edit_distance(str1, str2)
        score = similarity_score(str1, str2)
        print(f" * '{str1}' vs '{str2}': Khoảng cách = {ed} -> Độ tương đồng = {score:.2f}")

--- ĐANG CHẠY CÁC TEST CASES KIỂM TRA ---
Tất cả test cases phần (a) đều PASS!
--------------------------------------------------
Minh họa kết quả tính độ tương đồng similarity_score(s1, s2):
 * 'kitten' vs 'sitting': Khoảng cách = 3 -> Độ tương đồng = 0.57
 * 'sunday' vs 'saturday': Khoảng cách = 3 -> Độ tương đồng = 0.62
 * 'Nguyen' vs 'Nguyên': Khoảng cách = 1 -> Độ tương đồng = 0.83
 * 'abc' vs 'abc': Khoảng cách = 0 -> Độ tương đồng = 1.00
 * '' vs 'abc': Khoảng cách = 3 -> Độ tương đồng = 0.00


In [ ]:
def edit_distance(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j - 1],
                    dp[i - 1][j - 1])
    return dp[m][n]
def similarity_score(s1: str, s2: str) -> float:
    if not s1 and not s2:
        return 1.0
    max_len = max(len(s1), len(s2))
    return 1.0 - (edit_distance(s1, s2) / max_len)
def fuzzy_search(query: str, name_list: list, threshold: float = 0.75) -> list:
    results = []
    for name in name_list:
        score = similarity_score(query, name)
        if score >= threshold:
            results.append((name, score))
    results.sort(key=lambda x: x[1], reverse=True)
    return results
if __name__ == "__main__":
    file_path = "THONG_TIN_KHACH_HANG.json"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        all_names = [r["Họ và tên"] for r in data]

        queries = [
            "Nguyen Van Binh",
            "Tran Thi Lan",
            "Nguyễn Vân Bình",
            "Lê Minh Tấn",]
        for q in queries:
            results = fuzzy_search(q, all_names, threshold=0.7)
            print(f"\nQuery: '{q}'")
            if not results:
                print("  (Không tìm thấy kết quả nào thỏa mãn ngưỡng tương đồng)")
            else:
                for name, score in results[:5]:
                    print(f"  {score:.2f}  {name}")


Query: 'Nguyen Van Binh'
  (Không tìm thấy kết quả nào thỏa mãn ngưỡng tương đồng)

Query: 'Tran Thi Lan'
  (Không tìm thấy kết quả nào thỏa mãn ngưỡng tương đồng)

Query: 'Nguyễn Vân Bình'
  (Không tìm thấy kết quả nào thỏa mãn ngưỡng tương đồng)

Query: 'Lê Minh Tấn'
  (Không tìm thấy kết quả nào thỏa mãn ngưỡng tương đồng)


In [ ]:
def edit_distance(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]
def similarity_score(s1: str, s2: str) -> float:
    if not s1 and not s2: return 1.0
    max_len = max(len(s1), len(s2))
    return 1.0 - (edit_distance(s1, s2) / max_len)
def clean_phone(phone_str: str) -> str:
    if not phone_str: return ""
    digits = "".join(re.findall(r"\d+", str(phone_str)))
    if digits.startswith("84") and len(digits) > 9:
        digits = "0" + digits[2:]
    return digits
def clean_name_for_compare(name_str: str) -> str:
    if not name_str: return ""
    name_str = name_str.strip().replace("Đ", "D").replace("đ", "d")
    nfkd_form = unicodedata.normalize("NFKD", name_str)
    no_diacritics = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    return " ".join(no_diacritics.lower().split())
def combined_similarity(r1: dict, r2: dict) -> float:
    name1 = clean_name_for_compare(r1.get("Họ và tên", ""))
    name2 = clean_name_for_compare(r2.get("Họ và tên", ""))

    phone1 = clean_phone(r1.get("Số điện thoại", ""))
    phone2 = clean_phone(r2.get("Số điện thoại", ""))

    name_sim = similarity_score(name1, name2)
    phone_sim = similarity_score(phone1, phone2)

    return 0.6 * name_sim + 0.4 * phone_sim
def find_duplicates(records: list, threshold: float = 0.85) -> list:
    duplicates = []
    n = len(records)

    for i in range(n):
        for j in range(i + 1, n):
            score = combined_similarity(records[i], records[j])
            if score >= threshold:
                duplicates.append((records[i], records[j], score))

    duplicates.sort(key=lambda x: x[2], reverse=True)
    return duplicates
if __name__ == "__main__":
    file_path = "THONG_TIN_KHACH_HANG.json"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            full_dataset = json.load(f)

        sub_dataset = full_dataset[:100]
        print(f"-> Đang chạy kiểm tra trùng lặp trên {len(sub_dataset)} bản ghi đầu tiên...")

        duplicate_pairs = find_duplicates(sub_dataset, threshold=0.85)

        print("\n=== DANH SÁCH CÁC CẶP BẢN GHI NGHI NGỜ TRÙNG LẶP (>= 0.85) ===")
        if not duplicate_pairs:
            print("Không phát hiện cặp trùng lặp nào thỏa mãn điều kiện.")
        else:
            for idx, (r1, r2, score) in enumerate(duplicate_pairs, 1):
                print(f"Cặp trùng lặp #{idx} (Độ tương đồng: {score:.2%})")
                print(f"  [1] Mã KH: {r1.get('Mã KH'):9s} | Tên: {r1.get('Họ và tên'):20s} | SĐT: {r1.get('Số điện thoại')}")
                print(f"  [2] Mã KH: {r2.get('Mã KH'):9s} | Tên: {r2.get('Họ và tên'):20s} | SĐT: {r2.get('Số điện thoại')}")
                print("-" * 75)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file dữ liệu '{file_path}'.")

-> Đang chạy kiểm tra trùng lặp trên 100 bản ghi đầu tiên...

=== DANH SÁCH CÁC CẶP BẢN GHI NGHI NGỜ TRÙNG LẶP (>= 0.85) ===
Không phát hiện cặp trùng lặp nào thỏa mãn điều kiện.
